In [164]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [165]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [166]:
# Time
start = dt.datetime(2019,4,30)
end = dt.datetime(2019,5,2)
print(start,end,end-start)

2019-04-30 00:00:00 2019-05-02 00:00:00 2 days, 0:00:00


In [174]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}},{"sign_up_details":1, "created_at":1,"login_details":1}):
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
df_users = pd.DataFrame(dic_flattened)
df_users = df_users[df_users["sign_up_details_app_platform"] == "UNITY_Android"]
#df = df[df["sign_up_details_device_id"].isin(devices)]
users = df_users[["_id","created_at","sign_up_details_device_id","login_details_last_request_at"]]
users.columns = ["user_id","createtime","device_id","last_request"]
print(len(users))

359


In [176]:
users = users.sort_values(['device_id','createtime'])
users = users.drop_duplicates('device_id')

In [177]:
len(users)

356

In [178]:
team_cursor = cursor.superstars.teams
aw_teams = []
for documents in team_cursor.find({'created_at': {'$lt': end, '$gte': start}},{"user":1, "created_at":1}):
    aw_teams.append(documents)
dic_flattened = [flatten(d) for d in aw_teams]
df_teams = pd.DataFrame(dic_flattened)
teams = df_teams[df_teams["user"].isin(users["user_id"])]
teams = teams[["_id","user","created_at"]]
teams.columns = ["team_id", "user_id", "team_created_at"]
print(len(teams))
teams.head()

354


,team_id,user_id,team_created_at
0,5cc79098304ded1c8d247a1d,5cc79098304ded1c8d247a15,2019-04-30 00:02:32.279
1,5cc7986cc757b72b3a1c8689,5cc7986cc757b72b3a1c8681,2019-04-30 00:35:56.656
2,5cc798b6c1811e2b616907f7,5cc798b6c1811e2b616907ef,2019-04-30 00:37:10.234
3,5cc79982aaefa53d8f9ac78d,5cc79982aaefa53d8f9ac785,2019-04-30 00:40:34.013
4,5cc79b24c757b72b3a1c89f4,5cc79b24c757b72b3a1c89ec,2019-04-30 00:47:32.600


In [179]:
users_team =pd.merge(users[['user_id','createtime','device_id']],teams[['team_id','user_id']],on='user_id',how='inner')

In [180]:
users_team.head()

,user_id,createtime,device_id,team_id
0,5cc7eae9faa57c3d91d4b88d,2019-04-30 06:27:53.548,008fa700b776693cd021c68e814e8bde,5cc7eae9faa57c3d91d4b895
1,5cc80879304ded1c8d282057,2019-04-30 08:34:01.561,00a6ca02bf92a086a4817adde2ce90a2,5cc80879304ded1c8d28205f
2,5cc7bc12595eb03db6afcca3,2019-04-30 03:08:02.088,01045eadc84ee7cb8279f8461b54d897,5cc7bc12595eb03db6afccab
3,5cc7d8c5297df62b5aa06648,2019-04-30 05:10:29.191,021b61e099e99cc23eaf783d88aa4a27,5cc7d8c5297df62b5aa06650
4,5cc80430595eb03db6b252e1,2019-04-30 08:15:44.513,02cad8a46084612e98c6cfac5dc0aaa4,5cc80430595eb03db6b252e9


In [181]:
len(users_team)

354

In [182]:
con_cursor = cursor.superstars.matches
aw_matches = []
for documents in con_cursor.find({'created_at': {'$lt': end, '$gte': start},'status':3},
                                 {"home_team":1, "winner_team":1,"status":1,"type":1,"start_time":1}):
    aw_matches.append(documents)
dic_flattened = [flatten(d) for d in aw_matches]
df_matches = pd.DataFrame(dic_flattened)
df_matches = df_matches.rename(columns={'home_team_id':'team_id'})
matches = df_matches[df_matches["team_id"].isin(users_team["team_id"])]
matches = matches.loc[:,["team_id","type",'start_time']]
matches.head()

,team_id,type,start_time
1,5cc79098304ded1c8d247a1d,CAMPAIGN,2019-04-30 00:02:50.694
3,5cc79098304ded1c8d247a1d,CAMPAIGN,2019-04-30 00:05:31.634
5,5cc79098304ded1c8d247a1d,CAMPAIGN,2019-04-30 00:09:33.989
16,5cc79098304ded1c8d247a1d,CAMPAIGN,2019-04-30 00:19:30.573
23,5cc79098304ded1c8d247a1d,CAMPAIGN,2019-04-30 00:24:44.057


In [183]:
len(matches)

1013

In [184]:
users_matches = pd.merge(users_team[['createtime','device_id','team_id']],matches[['team_id','start_time']],on='team_id')

In [185]:
users_matches.head()

,createtime,device_id,team_id,start_time
0,2019-04-30 06:27:53.548,008fa700b776693cd021c68e814e8bde,5cc7eae9faa57c3d91d4b895,2019-04-30 06:28:26.369
1,2019-04-30 08:34:01.561,00a6ca02bf92a086a4817adde2ce90a2,5cc80879304ded1c8d28205f,2019-04-30 08:34:23.898
2,2019-04-30 05:10:29.191,021b61e099e99cc23eaf783d88aa4a27,5cc7d8c5297df62b5aa06650,2019-04-30 05:12:09.523
3,2019-04-30 15:41:30.589,060e8639096b8edd510e0bc20d71bb3c,5cc86caa76bdc61cb41ddd7d,2019-04-30 15:41:53.667
4,2019-04-30 15:41:30.589,060e8639096b8edd510e0bc20d71bb3c,5cc86caa76bdc61cb41ddd7d,2019-04-30 15:44:22.851


In [186]:
len(users_matches)

1013

In [187]:
users_matches = users_matches[(users_matches['start_time']-users_matches['createtime'])<'24:00:00']

In [190]:
len(users_matches)

869

In [191]:
users_matches.head()

,createtime,device_id,team_id,start_time
0,2019-04-30 06:27:53.548,008fa700b776693cd021c68e814e8bde,5cc7eae9faa57c3d91d4b895,2019-04-30 06:28:26.369
1,2019-04-30 08:34:01.561,00a6ca02bf92a086a4817adde2ce90a2,5cc80879304ded1c8d28205f,2019-04-30 08:34:23.898
2,2019-04-30 05:10:29.191,021b61e099e99cc23eaf783d88aa4a27,5cc7d8c5297df62b5aa06650,2019-04-30 05:12:09.523
3,2019-04-30 15:41:30.589,060e8639096b8edd510e0bc20d71bb3c,5cc86caa76bdc61cb41ddd7d,2019-04-30 15:41:53.667
4,2019-04-30 15:41:30.589,060e8639096b8edd510e0bc20d71bb3c,5cc86caa76bdc61cb41ddd7d,2019-04-30 15:44:22.851


In [193]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query = (
    f"""SELECT
    user_id as device_id,
  device.mobile_os_hardware_model as device
FROM `hitwicketsuperstars.analytics_190927423.events_2019*`
      WHERE _TABLE_SUFFIX BETWEEN '0430'
      AND '0502'
      GROUP BY user_id, device
      """
)
df_all = client.query(query).to_dataframe()

In [194]:
df_all.head()

,device_id,device
0,c059010c15da12a600fef87424fe5089,CPH1819
1,75654f364a81c90075daed924031bdca,RMX1801
2,None,SM-J200G
3,421790d81045bb4389dbbb6098bc0c39,RMX1831
4,cab2c643b4a74530bfc61ef40607132f,Lenovo TB3-710I


In [195]:
users_devices = pd.merge(users_matches[['device_id','start_time']],df_all,on='device_id',how='inner')

In [196]:
len(users_devices)

857

In [197]:
users_devices.head()

,device_id,start_time,device
0,008fa700b776693cd021c68e814e8bde,2019-04-30 06:28:26.369,Redmi 6A
1,00a6ca02bf92a086a4817adde2ce90a2,2019-04-30 08:34:23.898,CPH1819
2,021b61e099e99cc23eaf783d88aa4a27,2019-04-30 05:12:09.523,CPH1729
3,060e8639096b8edd510e0bc20d71bb3c,2019-04-30 15:41:53.667,moto g(6) play
4,060e8639096b8edd510e0bc20d71bb3c,2019-04-30 15:44:22.851,moto g(6) play


In [198]:
matches_count = users_devices.groupby('device_id')['start_time'].count().reset_index()
#matches_count.sort_values('start_time',ascending=True,inplace=True)
matches_count.columns = ['device_id','matches']
matches_count.head()

,device_id,matches
0,008fa700b776693cd021c68e814e8bde,1
1,00a6ca02bf92a086a4817adde2ce90a2,1
2,021b61e099e99cc23eaf783d88aa4a27,1
3,060e8639096b8edd510e0bc20d71bb3c,2
4,0667c2f5492df013f0a7f17eb10c4b4d,3


In [199]:
len(matches_count)

222

In [200]:
matches_count = pd.merge(matches_count,users[['device_id']],on='device_id',how='right')
matches_count = matches_count.fillna(0)
matches_count['matches'] = matches_count['matches'].astype(int)
len(matches_count)

356

In [201]:
matches_count.head()

,device_id,matches
0,008fa700b776693cd021c68e814e8bde,1
1,00a6ca02bf92a086a4817adde2ce90a2,1
2,021b61e099e99cc23eaf783d88aa4a27,1
3,060e8639096b8edd510e0bc20d71bb3c,2
4,0667c2f5492df013f0a7f17eb10c4b4d,3


In [202]:
matches_count['total_matches'] = np.where(matches_count['matches']>8,' 9+',matches_count['matches'])

In [203]:
matches_count.head()

,device_id,matches,total_matches
0,008fa700b776693cd021c68e814e8bde,1,1
1,00a6ca02bf92a086a4817adde2ce90a2,1,1
2,021b61e099e99cc23eaf783d88aa4a27,1,1
3,060e8639096b8edd510e0bc20d71bb3c,2,2
4,0667c2f5492df013f0a7f17eb10c4b4d,3,3


In [250]:
number_of_matches = matches_count.groupby('total_matches')['device_id'].count().reset_index()

In [251]:
df = number_of_matches
df = df.reindex(np.roll(df.index, shift=-1))

In [252]:
df.set_index('total_matches',inplace=True)

In [254]:
df['sum'] = [df.iloc[0][0]] + df.device_id[:0:-1].cumsum().values[::-1].tolist()

In [255]:
df

,device_id,sum
total_matches,,
0,134,134
1,86,222
2,54,136
3,26,82
4,13,56
5,7,43
6,6,36
7,2,30
8,3,28


In [256]:
df.device_id.sum()

356